# Introduction à pandas

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
from _data import load_accidents

## 1. Pourquoi pandas ?

En 2008, Wes McKinney travaille chez AQR Capital Management, un fonds d'investissement quantitatif.
Il manipule des séries temporelles financières en Python — et se heurte à un mur :
Python n'a pas d'équivalent au `data.frame` de R pour travailler avec des tables.
Il crée pandas pour combler ce manque. Le nom vient de *panel data*, terme économétrique
pour les données tabulaires avec une dimension temporelle.

Depuis, pandas est devenu **la lingua franca de la manipulation de données en Python**.
Quand un data scientist, un analyste ou un ingénieur parle de "faire du pandas",
tout le monde comprend : lire une table, la nettoyer, la transformer, l'agréger.

Ce que pandas apporte par rapport à Excel :
- **Scriptable** : chaque transformation est du code, pas des clics.
- **Reproductible** : relancer le script = même résultat.
- **Versionnable** : le code se gère avec git, pas le fichier `.xlsx`.
- **Scalable** : quelques millions de lignes, pas de problème (au-delà, il existe d'autres outils — on y vient).

## 2. La data stack moderne en 2026

pandas n'est plus seul. Voici les outils que vous allez croiser en entreprise :

| Outil | En une phrase | Quand l'utiliser |
|---|---|---|
| **pandas** | La référence interactive, lingua franca. | Toujours : c'est ce qu'on apprend ici. |
| **polars** | Successeur moderne, API chaînée, moteur Rust, lazy par défaut. | Quand pandas rame sur vos données. |
| **DuckDB** | "SQLite pour l'analytique" — SQL directement sur des DataFrames ou des fichiers Parquet. | Quand vous pensez naturellement en SQL ou que vos données sont en fichiers. |
| **PySpark** | Calcul distribué sur cluster (JVM). Dominant pendant 10 ans, moins incontournable qu'avant. | Quand les données ne tiennent plus sur une machine. |
| **dask** | Parallélisme pandas-like, sans cluster. | Volumes intermédiaires (10 Go – 1 To) sur une seule machine. |
| **ibis** | DSL unifié : même code Python, backend swappable (DuckDB, BigQuery, Spark…). | Quand vous voulez écrire du code portable entre environnements. |
| **narwhals** | API compatible pandas et polars pour les librairies qui veulent supporter les deux. | Développement de librairies, pas d'analyse directe. |
| **modin** | Drop-in replacement de pandas, parallélisme transparent. | Tentative de garder l'API pandas en accélérant les calculs. |

> **Message à retenir** : pandas reste le point d'entrée, mais vous allez croiser ces outils en entreprise.
> Le notebook 11 vous montrera concrètement la même analyse dans plusieurs de ces outils côte à côte.

## 3. Premier contact avec un DataFrame

On charge le dataset qui accompagne tout ce module : les accidents corporels de la route en France en 2024
(source : data.gouv.fr). C'est un vrai dataset de production — pas un jouet pédagogique.

In [ ]:
df = load_accidents()
df.shape  # (nombre de lignes, nombre de colonnes)

In [ ]:
df.head()

In [ ]:
# info() : types, valeurs manquantes, mémoire — à appeler systématiquement sur un nouveau dataset
df.info()

In [ ]:
# describe() : statistiques descriptives sur les colonnes numériques
df.describe()

Quatre commandes, et vous avez déjà une image du dataset :
combien de lignes, quels types, y a-t-il des valeurs manquantes, quelle est la distribution des nombres.

C'est le réflexe numéro un à avoir sur n'importe quel nouveau dataset.

## 4. Cinq choses que pandas sait faire

Un aperçu rapide — chaque exemple sera détaillé dans le notebook dédié.

In [ ]:
# 1. Compter — distribution d'une variable catégorielle
# col : type de collision (1=frontale, 2=arrière, 3=côté, 4=en chaîne, 5=multiple, 6=autre, 7=sans collision)
# On verra .value_counts() en détail dans le notebook 4.0 (groupby).

df["col"].value_counts()

In [ ]:
# 2. Filtrer — garder uniquement les accidents parisiens
# On verra .query() en détail dans le notebook 3 (sélectionner et filtrer).

df.query("dep == 75").shape

In [ ]:
# 3. Grouper — top 10 des départements par nombre d'accidents
# On verra .groupby() en détail dans le notebook 4.0.

df.groupby("dep")["Num_Acc"].count().nlargest(10)

In [ ]:
# 4. Tracer — nombre d'accidents par mois
# On verra la visualisation en détail dans le module plot/.

(
    df
    .groupby("mois")["Num_Acc"]
    .count()
    .plot(kind="bar", title="Accidents par mois (2024)", xlabel="Mois", ylabel="Nombre d'accidents")
)

In [ ]:
# 5. Lire / écrire — exporter en Parquet (format moderne, compact, rapide)
# On verra les formats en détail dans le notebook 8.

df.to_parquet("accidents.parquet")
print("Fichier écrit.")

# Relire :
df_relu = pd.read_parquet("accidents.parquet")
df_relu.shape

## 5. Ce que vous verrez dans ce module

### Parcours principal (Jour 2)

| Notebook | Sujet |
|---|---|
| **2** — La DataFrame et la Série | Structures fondamentales, arithmétique vectorisée |
| **3** — Sélectionner et filtrer | `.query()`, `[]`, `.loc[]` — court et pratique |
| **9** — Style moderne | Method chaining : `.assign()`, `.pipe()`, `.query()` |
| **4.0** — Groupby | Split-apply-combine, `.agg()` |
| **4.1** — Combining | `merge`, `concat` sur les 4 tables accidents |
| **4.2** — Reshaping | `pivot_table`, `melt`, `crosstab` |
| **5** — apply / map / transform | Appliquer des fonctions, quand éviter `apply` |
| **6** — Strings | `.str.contains()`, `.str.replace()`, `.str.split()` |
| **7** — Les types | `dtype`, valeurs manquantes, pandas 2.0 |
| **8** — Lire et enregistrer | CSV, Parquet, Excel — règles de choix |
| **10** — Time series | Dates, resampling, rolling |
| **11** — Écosystème | pandas vs polars vs DuckDB vs Spark |

### Approfondissements (`advance/`)

Pour ceux qui veulent creuser après le cours :
- `3.1` — Subtilités `.loc` / `.iloc`
- `3.2` — `.where()`, masques combinés, cas limites
- `performance_categoriels` — types catégoriels, mémoire, vitesse
- `styling_dataframe` — mise en forme des DataFrames pour l'export

## Quiz d'orientation

<details><summary>Vous avez 800 Go de données sur un laptop avec 32 Go de RAM. Quel outil ?</summary>

**DuckDB** ou **dask**. DuckDB peut interroger des fichiers Parquet sans tout charger en mémoire
(il lit colonne par colonne, filtre en streaming). dask fait la même chose avec une API pandas-like.
polars en mode lazy est aussi une option si les données tiennent dans un seul fichier.
pandas tout seul : non, il chargera tout en RAM.

</details>

<details><summary>Votre équipe travaille sur un cluster Databricks. Quel outil ?</summary>

**PySpark** — c'est le moteur natif de Databricks. Vous pouvez aussi utiliser pandas sur Databricks
pour les petits datasets (via `pandas on Spark` / `pyspark.pandas`), mais dès que vous utilisez
le cluster distribué, c'est PySpark.

</details>

<details><summary>Vous connaissez bien SQL et vos données sont des fichiers Parquet sur S3. Quel outil ?</summary>

**DuckDB**. Il peut requêter directement des fichiers Parquet locaux ou sur S3 avec du SQL pur,
sans infrastructure à monter. `duckdb.sql("SELECT ... FROM 'fichier.parquet'")` — c'est tout.

</details>